<a href="https://colab.research.google.com/github/msiplab/VieWork/blob/master/code/python/vie_sec03_color.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://matlab.mathworks.com/open/github/v1?repo=msiplab/VieWork&file=code/matlab/vie_sec03_color.m&project=VieWork.prj" target="_parent"><img src="https://www.mathworks.com/images/responsive/global/open-in-matlab-online.svg" alt="Open in MATLAB Online"/></a>

# 第3回 画像表現と色空間
村松正吾　「画像情報工学」

動作確認：Python 3.12（Google Colab），NumPy，scikit-image，Matplotlib，Pillow

MATLAB のライブスクリプト `code/matlab/vie_sec03_color.m` を Python に移植したもの。第3回のスライド（vie2026-03）で使う図と数値を作る。量子化，データ量（静止画像・映像），RGB カラー方式，色空間（RGB, XYZ, YCbCr, CMY, CMYK, HSV），インデックス方式を順に扱う。記号は教科書に合わせ，一画素当たりのビット数を $\beta$，総ビット数を $B$，ビットレートを $R$，量子化ステップを $Q$，フレーム間隔を $\Delta_\mathrm{t}$ とする。

スライドの図と数値は MATLAB 版で作っている。新潟大学の学生は[大学の包括ライセンス](https://jp.mathworks.com/academia/tah-portal/niigata-university-31576302.html)で MATLAB を使えるので，MATLAB 版もぜひ動かしてほしい。上の「Open in MATLAB Online」のボタンを押すと，ブラウザだけで MATLAB Online が開き，VieWork が自分の MATLAB Drive に複製されて，このノートブックに対応するライブスクリプトが開く（インストールは不要）。さらに [MATLAB Connector](https://jp.mathworks.com/products/matlab-drive.html) を手元の PC に入れると，MATLAB Drive のファイルが PC のフォルダと自動で同期されるので，MATLAB Online で作業したファイルを手元の MATLAB でもそのまま開ける。

スライドに載せる数値は，すべてこの中で計算して表示する。画像処理には scikit-image，NumPy，Pillow を使う。教科書の例・例題の数値は，ここで計算し直して教科書（スライド）の値と一致することを `assert` で確かめる。画像から計算する値は，ライブラリの実装が MATLAB と異なるため，スライドの値とわずかに異なることがある。

## 準備
共通ヘルパー `vie.py`（サンプル画像の取得，日本語フォントと配色の設定）を読み込む。Colab では GitHub から取得する。

In [ ]:
import importlib
import os
import sys
import urllib.request

# Colab では毎回 GitHub から最新の vie.py を取得する（VieWork を clone した場合は同じフォルダの vie.py を使う）
if "google.colab" in sys.modules or not os.path.exists("vie.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/msiplab/VieWork/master/code/python/vie.py", "vie.py")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.path import Path
from PIL import Image
from skimage import color
from skimage.metrics import peak_signal_noise_ratio
from skimage.transform import rescale, resize
from skimage.util import img_as_float, img_as_ubyte
import vie

vie = importlib.reload(vie)   # 取得し直した vie.py を読み込み直す

# 図の線や点の色は，スライドのロゴの 3 色（メインの緑，青，橙）にそろえる。
# 色そのものが内容の図（成分画像，色空間の図）は除く
cmain, ccool, cwarm, cgray = vie.CMAIN, vie.CCOOL, vie.CWARM, vie.CGRAY


def show(ax, X, title=None):
    """画像を軸なし・縦横比 1 で表示する（グレースケールは 0 を黒，最大値（255 か 1）を白）．"""
    vmax = 255 if X.dtype == np.uint8 else 1
    ax.imshow(X, cmap="gray", vmin=0, vmax=vmax, interpolation="nearest")
    ax.set_axis_off()
    if title is not None:
        ax.set_title(title)

写真は教科書（参考資料）のサンプル画像 msipimg02（花束，512×512 画素）を使う。色とりどりの花が写っていて，RGB の各成分，HSV の各成分，減色の効果がはっきり見える。スライドの図の配置に合わせて，中央付近の $342\times 512$ 画素（行 139〜480）を切り出して横長にし，$256\times 384$ 画素に縮小する。

In [ ]:
Xrgb = img_as_float(vie.msipimg(2))           # 花束（RGB，512×512）
Xrgb = Xrgb[139:481]                          # 横長（縦:横 = 2:3）に切り出す
Xrgb = resize(Xrgb, (256, 384), order=3, anti_aliasing=True)   # 256×384 画素に縮小（スライド用）
N1, N2 = Xrgb.shape[:2]
print("N1 =", N1, " N2 =", N2)

## 画像の量子化：輝度値の離散化
8-bit の輝度値 $x\in\{0,1,\dots,255\}$ を量子化ステップ $Q$ で離散化する。前回スライドと同じく小数点以下切り捨てを用いる（教科書の線形量子化では四捨五入 $\lfloor \cdot \rceil$ も使える）。

$$y = \left\lfloor \frac{x}{Q} \right\rfloor$$

階調数を $L$ とすると $Q = 256/L$ である。まず簡単な数値で確かめる。$L=4$ なら $Q=64$。

In [ ]:
L = 4
Q = 256 // L
x = np.array([37, 100, 200, 255])
y = x // Q
print("Q =", Q)
print("x =", x)
print("y =", y)
assert Q == 64 and np.array_equal(y, [0, 1, 3, 3])   # スライドの値

入出力の関係（量子化特性）は階段状になる。数値例はスライドの例の枠に書くので，図には重ねない。

In [ ]:
xx = np.arange(256)
fig, ax = plt.subplots(figsize=(4.5, 3.4), layout="constrained")
ax.step(xx, xx // Q, where="post", color=cmain, lw=1.5)
ax.grid(True)
ax.set(xlim=(0, 256), ylim=(-0.2, L - 0.8), xticks=np.arange(0, 257, Q), yticks=np.arange(L))
ax.set_xlabel("$x$", fontsize=12)
ax.set_ylabel(r"$y=\lfloor x/Q\rfloor$", fontsize=12)
ax.tick_params(labelsize=11)
plt.show()

## 階調数を変えた画像
グレースケール画像を $L=2,4,256$ 階調に量子化する。表示のため $y/(L-1)$ で 0〜1 に戻す。階調が少ないと擬似輪郭が目立つ。

In [ ]:
Xg = img_as_ubyte(np.clip(Xrgb @ np.array([0.299, 0.587, 0.114]), 0, 1))   # 輝度 Y（教科書の YCbCr の Y）
fig, axs = plt.subplots(1, 3, figsize=(12, 2.9), layout="constrained")
for ax, L in zip(axs, [2, 4, 256]):
    Q = 256 // L
    Yq = Xg.astype(np.float64) // Q           # 量子化インデックス 0〜L-1
    show(ax, Yq / (L - 1), f"L = {L}")
plt.show()

## ディジタル画像のデータ量（グレースケール）
階調数 $L$ を表すのに必要な一画素当たりのビット数は $\beta = \lceil \log_2 L \rceil$。計算の都合上，$L$ は通常 2 のべき乗に選ぶ。階調数とビット数の対応表をそのままスライドの表にする。

In [ ]:
Ls = np.array([2, 4, 256])
betaL = np.ceil(np.log2(Ls)).astype(int)
print("階調数 L    :", *Ls)
print("β [bpp]     :", *betaL)
assert np.array_equal(betaL, [1, 2, 8])

$N_1\times N_2$ 画素の総ビット数は $B=\beta N_1 N_2$。例として $1080\times 1920$ 画素，$L=256$ 階調（$\beta=8$）の場合：

In [ ]:
Nv, Nh, betaPx = 1080, 1920, 8
Npix = Nv * Nh                                # 総画素数 N1 N2
Bgray = betaPx * Npix                         # [bits]
Bbytes = Bgray / 8                            # [bytes]
print(f"Npix = {Npix:,}，Bgray = {Bgray:,} bits，Bbytes = {Bbytes:,.0f} bytes ≃ {Bbytes / 1e6:.1f} MB")

## 教科書の例「静止画像のデータ量」
$N_1\times N_2=2304\times 3456$ 画素（約 800 万画素）の二種類の画像の総ビット数を計算する。
- 8-bit 符号なし整数型（$\beta=8$ bits）のグレースケール画像：$B=\beta N_1N_2$
- 倍精度実数型（$\beta=64$ bits）の RGB カラー画像：$B=3\beta N_1N_2$

MB は $10^6$ bytes，1 byte = 8 bits として換算する。

In [ ]:
N1s, N2s = 2304, 3456
Bs8 = 8 * N1s * N2s                           # グレースケール（uint8）[bits]
Bs64 = (3 * 64) * N1s * N2s                   # RGB（double）[bits]
MBs = np.array([Bs8, Bs64]) / 8 / 1e6         # [MB]
ratioS = Bs64 // Bs8                          # 後者は前者の何倍か
print(f"Bs8 = {Bs8:,} bits（{MBs[0]:.0f} MB），Bs64 = {Bs64:,} bits（{MBs[1]:.0f} MB），{ratioS} 倍")
assert (Bs8, Bs64) == (63700992, 1528823808)  # 教科書の値と一致するか

実際の画像配列の大きさ（バイト数）は `nbytes` で確かめられる（講義のデモ dataamount で MATLAB の `whos` を使うのと同じ）。8-bit の RGB 画像なら $3N_1N_2$ bytes になる。

In [ ]:
X8bit = img_as_ubyte(Xrgb)
print(X8bit.nbytes, "bytes")
assert X8bit.nbytes == 3 * N1 * N2

## ディジタル映像のデータ量（グレースケール）
フレーム間隔を $\Delta_\mathrm{t}$ とすると，ビットレートは $R = \beta N_1 N_2 \Delta_\mathrm{t}^{-1}$ [bps]。$\Delta_\mathrm{t} = 1/30$ s の場合（$1/30$ は二進の浮動小数点数で割り切れないので，フレームレート $\Delta_\mathrm{t}^{-1}=30$ を掛ける）：

In [ ]:
fr = 30                                       # フレームレート 1/Δt [1/s]
Rgray = betaPx * Nv * Nh * fr                 # [bps]
print(f"Rgray = {Rgray:,} bps ≃ {Rgray / 1e6:.1f} Mbps")

## RGB カラー方式
各成分（R, G, B）をそれぞれ配列で表す。総ビット数は $B=(\beta_\mathrm{R}+\beta_\mathrm{G}+\beta_\mathrm{B})N_1N_2$。

In [ ]:
Z = np.zeros((N1, N2))
fig, axs = plt.subplots(1, 4, figsize=(12, 2.3), layout="constrained")
show(axs[0], np.stack([Xrgb[:, :, 0], Z, Z], axis=2), "R")
show(axs[1], np.stack([Z, Xrgb[:, :, 1], Z], axis=2), "G")
show(axs[2], np.stack([Z, Z, Xrgb[:, :, 2]], axis=2), "B")
show(axs[3], Xrgb, "カラー")
plt.show()

この画像（$256\times 384$ 画素，各成分 8 bit）の総ビット数：

In [ ]:
Brgb = (8 + 8 + 8) * N1 * N2
print(f"{N1}×{N2} 画素，Brgb = {Brgb:,} bits")

## RGB カラー方式と量子化
各成分を 1 bit にすると表せる色は $2^3=8$ 色，8 bit なら $256^3$ 色。

In [ ]:
ncolors = [2 ** 3, 256 ** 3]
print(f"ncolors = {ncolors[0]}，{ncolors[1]:,}")
X8 = (Xrgb >= 0.5).astype(np.float64)         # 各成分を 1 bit に量子化
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), layout="constrained")
show(axs[0], X8, "8 色")
show(axs[1], Xrgb, "16,777,216 色")
plt.show()

画素の色を RGB 空間 $[0,1]^3$ の点として描く（講義のデモ rgbcolordemo と同じ見せ方）。各成分 1 bit の画像では，画素は立方体の 8 つの頂点にしか現れない。各成分 8 bit の画像では，画素が立方体の中に雲のように広がる。点の色はその画素の色である。

In [ ]:
def plotrgbcloud(ax, X, msz):
    """画像 X（N1×N2×3，値は 0〜1）の画素の色を RGB 空間の点として描く．

    点の色はその画素の色．重なる色は 1 点にまとめ，多すぎるときは間引く．
    """
    c = np.unique(X.reshape(-1, 3), axis=0)
    if c.shape[0] > 8000:
        c = c[np.round(np.linspace(0, c.shape[0] - 1, 8000)).astype(int)]
    V = np.array([[0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0],
                  [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1]])   # 立方体の頂点
    E = [(0, 1), (1, 2), (2, 3), (3, 0), (4, 5), (5, 6), (6, 7), (7, 4),
         (0, 4), (1, 5), (2, 6), (3, 7)]                          # 立方体の辺
    for i, j in E:
        ax.plot(*V[[i, j]].T, color=cgray, lw=0.8)
    ax.plot([0, 1], [0, 1], [0, 1], "--", color=cgray, lw=0.8)  # 黒と白を結ぶ対角線（グレースケール）
    for k in range(3):                         # 黒から出る 3 辺を R, G, B の座標軸として濃く描く
        e = np.zeros((2, 3))
        e[1, k] = 1
        ax.plot(*e.T, color=(0.2, 0.2, 0.2), lw=1)
    edgecolor = (0.4, 0.4, 0.4) if msz > 20 else "none"   # 大きな点は縁取り（白い点も見えるように）
    ax.scatter(c[:, 0], c[:, 1], c[:, 2], s=msz, c=np.clip(c, 0, 1), edgecolors=edgecolor,
               depthshade=False)
    ax.set(xlim=(-0.06, 1.06), ylim=(-0.06, 1.06), zlim=(-0.06, 1.06))  # 頂点の点が切れないよう少し広げる
    ax.set_box_aspect((1, 1, 1))
    ax.set_axis_off()
    ax.view_init(elev=25, azim=-37.5 - 90)     # 仰角 25°，R 軸の負側から 37.5° 回った向きから見る
    ax.set_proj_type("ortho")                  # 平行投影
    ax.text(0.55, -0.12, -0.06, r"$x_\mathrm{R}$", fontsize=14, ha="left", va="top")
    ax.text(-0.12, 0.55, -0.06, r"$x_\mathrm{G}$", fontsize=14, ha="right", va="top")
    ax.text(-0.08, -0.08, 0.5, r"$x_\mathrm{B}$", fontsize=14, ha="right")


fig = plt.figure(figsize=(9, 4.5), layout="constrained")
ax1 = fig.add_subplot(1, 2, 1, projection="3d")
plotrgbcloud(ax1, X8, 60)
ax1.set_title("8 色")
ax2 = fig.add_subplot(1, 2, 2, projection="3d")
plotrgbcloud(ax2, Xrgb, 3)
ax2.set_title("16,777,216 色")
plt.show()

## 【例】ディジタル映像のデータ量（RGB）
$1080\times 1920$ 画素，各フレーム RGB（$\beta_\mathrm{R}=\beta_\mathrm{G}=\beta_\mathrm{B}=8$），$\Delta_\mathrm{t}=1/30$ s のビットレート（教科書の章末問題と同じ設定）：

In [ ]:
Rrgb = (8 + 8 + 8) * Nv * Nh * fr             # [bps]
print(f"Rrgb = {Rrgb:,} bps ≃ {Rrgb / 1e9:.3f} Gbps")

## 教科書の例題「動画像のビットレート」
RGB カラー動画像，$N_1\times N_2=4320\times 7680$ 画素，$\Delta_\mathrm{t}=1/60$ s，$\beta=8$ bits。一画素当たり $3\beta$ bpp，1 フレーム当たり $B=3\beta N_1N_2$ bits，ビットレートは $R=B\Delta_\mathrm{t}^{-1}$。

In [ ]:
N1u, N2u, fru, betaU = 4320, 7680, 60, 8      # fru はフレームレート 1/Δt
Nu = N1u * N2u                                # 1 フレームの画素数
bppU = 3 * betaU                              # [bpp]
Bu = bppU * Nu                                # [bits/frame]
Ru = Bu * fru                                 # [bps]
print(f"{bppU} bpp，Nu = {Nu:,}，Bu = {Bu:,} bits/frame，Ru = {Ru:,} bps ≃ {Ru / 1e9:.0f} Gbps")
assert (Nu, Bu, Ru) == (33177600, 796262400, 47775744000)   # 教科書の値と一致するか

## RGB 空間
RGB の各成分を座標軸にとると，色は単位立方体 $[0,1]^3$ の中の点になる。黒 (0,0,0) と白 (1,1,1) を結ぶ対角線がグレースケール。

立方体の表面の点の色はその点の座標そのものなので，見える 3 つの面（$x_\mathrm{R}=1$，$x_\mathrm{G}=1$，$x_\mathrm{B}=1$）を細かい格子に分けて，各点をその座標の色で塗る。

In [ ]:
fig = plt.figure(figsize=(4, 4), layout="constrained")
ax = fig.add_subplot(projection="3d", computed_zorder=False)   # 辺を面の上に描く
t = np.linspace(0, 1, 33)
u, v = np.meshgrid(t, t)
one = np.ones_like(u)
for P in ((one, u, v), (u, one, v), (u, v, one)):   # x_R = 1，x_G = 1，x_B = 1 の面
    ax.plot_surface(*P, facecolors=np.stack(P, axis=2), shade=False,
                    rstride=1, cstride=1, linewidth=0, antialiased=False)
V = np.array([[0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0],
              [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1]])
for i, j in [(1, 2), (2, 3), (4, 5), (5, 6), (6, 7), (7, 4), (1, 5), (2, 6), (3, 7)]:   # 見える辺
    ax.plot(*V[[i, j]].T, color="k", lw=1, zorder=10)
ax.set_box_aspect((1, 1, 1))
ax.set_axis_off()
ax.view_init(elev=np.degrees(np.arctan2(0.9, np.hypot(1, 0.8))),
             azim=np.degrees(np.arctan2(0.8, 1)))   # 方向 (1, 0.8, 0.9) から見る
ax.set_proj_type("ortho")                           # 平行投影
plt.show()

グレースケール変換（BT.601）の係数 $a_0=0.299,\ a_1=0.587,\ a_2=0.114$ は和が 1 になる。

In [ ]:
a601 = np.array([0.299, 0.587, 0.114])
print(a601.sum())

## xy 色度図
CIE の XYZ 空間から $x=X/S,\ y=Y/S\ (S=X+Y+Z)$ として得た xy 色度図。三角形は sRGB の三原色で表せる色の範囲。

CIE 1931 等色関数の数表（波長 400〜700 nm，5 nm 刻み）からスペクトル軌跡を求め，その内側をその色度の色で塗る関数を定義して描く。

In [ ]:
# CIE 1931 等色関数（2 度視野）の値．波長 400〜700 nm，5 nm 刻み（CIE の数表）
cmf_x = np.array([
    0.01431, 0.02319, 0.04351, 0.07763, 0.13438, 0.21477, 0.2839, 0.3285, 0.34828, 0.34806,
    0.3362, 0.3187, 0.2908, 0.2511, 0.19536, 0.1421, 0.09564, 0.05795, 0.03201, 0.0147, 0.0049,
    0.0024, 0.0093, 0.0291, 0.06327, 0.1096, 0.1655, 0.22575, 0.2904, 0.3597, 0.43345, 0.51205,
    0.5945, 0.6784, 0.7621, 0.8425, 0.9163, 0.9786, 1.0263, 1.0567, 1.0622, 1.0456, 1.0026,
    0.9384, 0.85445, 0.7514, 0.6424, 0.5419, 0.4479, 0.3608, 0.2835, 0.2187, 0.1649, 0.1212,
    0.0874, 0.0636, 0.04677, 0.0329, 0.0227, 0.01584, 0.011359])
cmf_y = np.array([
    0.000396, 0.00064, 0.00121, 0.00218, 0.004, 0.0073, 0.0116, 0.01684, 0.023, 0.0298, 0.038,
    0.048, 0.06, 0.0739, 0.09098, 0.1126, 0.13902, 0.1693, 0.20802, 0.2586, 0.323, 0.4073,
    0.503, 0.6082, 0.71, 0.7932, 0.862, 0.91485, 0.954, 0.9803, 0.99495, 1, 0.995, 0.9786,
    0.952, 0.9154, 0.87, 0.8163, 0.757, 0.6949, 0.631, 0.5668, 0.503, 0.4412, 0.381, 0.321,
    0.265, 0.217, 0.175, 0.1382, 0.107, 0.0816, 0.061, 0.04458, 0.032, 0.0232, 0.017, 0.01192,
    0.00821, 0.005723, 0.004102])
cmf_z = np.array([
    0.06785, 0.1102, 0.2074, 0.3713, 0.6456, 1.03905, 1.3856, 1.62296, 1.74706, 1.7826,
    1.77211, 1.7441, 1.6692, 1.5281, 1.28764, 1.0419, 0.81295, 0.6162, 0.46518, 0.3533, 0.272,
    0.2123, 0.1582, 0.1117, 0.07825, 0.05725, 0.04216, 0.02984, 0.0203, 0.0134, 0.00875,
    0.00575, 0.0039, 0.00275, 0.0021, 0.0018, 0.00165, 0.0014, 0.0011, 0.001, 0.0008, 0.0006,
    0.00034, 0.00024, 0.00019, 0.0001, 0.00005, 0.00003, 0.00002, 0.00001, 0, 0, 0, 0, 0, 0, 0,
    0, 0, 0, 0])


def plot_chromaticity(ax, n=400):
    """xy 色度図を描く（スペクトル軌跡の内側を，その色度の色で塗る）．"""
    XYZ = np.stack([cmf_x, cmf_y, cmf_z], axis=1)
    locus = XYZ[:, :2] / XYZ.sum(axis=1, keepdims=True)    # スペクトル軌跡の xy
    g = (np.arange(n) + 0.5) / n * 0.85
    gx, gy = np.meshgrid(g, g)
    inside = Path(locus).contains_points(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)
    # 色度 (x, y) の XYZ（Y = 1）を線形 sRGB に変換し，最大の成分が 1 になるよう明るさをそろえる
    # XYZ から線形 sRGB への変換行列（下の「XYZ 空間」の節で求める行列 A の逆行列）
    M = np.array([[3.2406,-1.5372, -0.4986], [-0.9689, 1.8758, 0.0415], [0.0557, -0.2040, 1.0570]])
    xyz = np.stack([gx / gy, np.ones_like(gx), (1 - gx - gy) / gy], axis=2)
    lin = np.clip(xyz @ M.T, 0, None)
    lin = lin / lin.max(axis=2, keepdims=True)
    rgb = np.where(lin <= 0.0031308, 12.92 * lin, 1.055 * lin ** (1 / 2.4) - 0.055)   # sRGB のガンマ
    img = np.ones((n, n, 4))
    img[..., :3] = rgb
    img[..., 3] = inside
    ax.imshow(img, origin="lower", extent=(0, 0.85, 0, 0.85), interpolation="bilinear")
    ax.plot(*np.vstack([locus, locus[:1]]).T, color=(0.3, 0.3, 0.3), lw=0.8)
    ax.set(xlim=(0, 0.85), ylim=(0, 0.85))
    ax.set_aspect("equal")
    ax.grid(True)


fig, ax = plt.subplots(figsize=(4, 4), layout="constrained")
plot_chromaticity(ax)
prim = np.array([[0.64, 0.33], [0.30, 0.60], [0.15, 0.06]])   # sRGB の R, G, B 原色の xy
ax.plot(*prim[[0, 1, 2, 0]].T, "k-", lw=1.5)
ax.set(xticks=np.arange(0, 0.81, 0.2), yticks=np.arange(0, 0.81, 0.2))
ax.set_xlabel("$x$", fontsize=12)
ax.set_ylabel("$y$", fontsize=12)
plt.show()

## XYZ 空間（教科書の例「XYZ空間」）
線形 RGB 空間の画素 $\vec{x}=(x_\mathrm{R}\ x_\mathrm{G}\ x_\mathrm{B})^\top$ から XYZ 画素 $\vec{y}=(y_\mathrm{X}\ y_\mathrm{Y}\ y_\mathrm{Z})^\top$ への線形変換 $\vec{y}=\boldsymbol{A}\vec{x}$ の行列を，sRGB の三原色の色度と白色点 D65 の色度 $(0.3127,\ 0.3290)$ から求める。

色度 $(x,y)$ の色の XYZ は $(x/y,\ 1,\ (1-x-y)/y)$ の定数倍である。原色 R, G, B の XYZ をこの形の列ベクトルの $s_k$ 倍とし，白 $\vec{x}=(1\ 1\ 1)^\top$ が D65 の XYZ（$Y=1$）に写るように倍率 $s_k$ を決める。

In [ ]:
wD65 = np.array([[0.3127, 0.3290]])           # D65 の色度 (x, y)


def xyz2col(c):
    """色度 (x, y) の各行を，Y = 1 に正規化した XYZ の列ベクトルにする．"""
    return np.stack([c[:, 0] / c[:, 1], np.ones(c.shape[0]), (1 - c[:, 0] - c[:, 1]) / c[:, 1]])


Cp = xyz2col(prim)                            # 列が原色の XYZ（Y=1 に正規化）
w = xyz2col(wD65)                             # D65 の XYZ（Y=1）
s = np.linalg.solve(Cp, w).ravel()            # 各原色の倍率
Axyz = Cp @ np.diag(s)
print("Axyz =\n", Axyz)

小数第 4 位に丸めると教科書の行列と一致する。2 行目（Y 成分）の和は 1 で，白の輝度が 1 になる。

In [ ]:
Axyz4 = np.round(Axyz, 4)
print("Axyz4 =\n", Axyz4)
print("2 行目の和:", Axyz[1].sum())
Axyz_book = np.array([[0.4124, 0.3576, 0.1805], [0.2126, 0.7152, 0.0722], [0.0193, 0.1192, 0.9505]])
assert np.array_equal(Axyz4, Axyz_book)       # 教科書の値と一致するか

## YCbCr 空間（BT.601）
教科書の例「YCbCr空間」は，ガンマ補正済みの $x_\mathrm{R'},x_\mathrm{G'},x_\mathrm{B'}\in[0,1]$ から

$$\begin{pmatrix} y_\mathrm{Y'} \\ y_\mathrm{Cb} \\ y_\mathrm{Cr} \end{pmatrix} \simeq \frac{1}{256}\begin{pmatrix} 65.7380 & 129.0570 & 25.0640 \\ -37.9450 & -74.4940 & 112.4390 \\ 112.4390 & -94.1540 & -18.2850 \end{pmatrix}\begin{pmatrix} x_\mathrm{R'} \\ x_\mathrm{G'} \\ x_\mathrm{B'} \end{pmatrix} + \begin{pmatrix} 16/255 \\ 128/255 \\ 128/255 \end{pmatrix}$$

とする。全体を 255 倍すると，8-bit（0〜255）の R'G'B' から 8-bit の Y'CbCr への変換になる（行列は同じで，オフセットが $(16\ 128\ 128)^\top$）。スライドの行列もここから表示する。

In [ ]:
Mycc = np.array([[65.738, 129.057, 25.064], [-37.945, -74.494, 112.439], [112.439, -94.154, -18.285]])
A = Mycc / 256
b = np.array([[16], [128], [128]])
np.set_printoptions(precision=4, suppress=True)
print("Mycc =\n", Mycc)

1 行目を $255/219$ 倍すると BT.601 の輝度の係数 $(0.299\ 0.587\ 0.114)$ に戻る（Y' の範囲が 16〜235 の 219 段階であるため）。

In [ ]:
print(A[0] * 255 / 219)
A3 = np.round(A, 3)                           # 前回スライドの係数（小数第 3 位）
print("A3 =\n", A3)
assert np.array_equal(A3, [[0.257, 0.504, 0.098], [-0.148, -0.291, 0.439], [0.439, -0.368, -0.071]])

代表的な色で確かめる。列は赤，緑，青，白，黒。BT.601 の定義（輝度 $Y'=0.299x_\mathrm{R'}+0.587x_\mathrm{G'}+0.114x_\mathrm{B'}$ を 16〜235 の 219 段階に，色差 $(x_\mathrm{B'}-Y')/1.772$，$(x_\mathrm{R'}-Y')/1.402$ を 16〜240 の 224 段階に割り当てる）から変換する関数を定義し，その結果とも一致することを確かめる。

In [ ]:
a601 = np.array([0.299, 0.587, 0.114])
T601 = np.vstack([219 * a601,                                   # Y'
                  224 * (np.array([0, 0, 1]) - a601) / 1.772,   # Cb
                  224 * (np.array([1, 0, 0]) - a601) / 1.402])  # Cr
off601 = np.array([16, 128, 128])


def rgb2ycbcr(X):
    """BT.601 の R'G'B'（0〜1 の実数）→ Y'CbCr（0〜1 の実数．8 bit なら 255 倍）．"""
    return X @ (T601 / 255).T + off601 / 255


def ycbcr2rgb(Y):
    """BT.601 の Y'CbCr（0〜1 の実数）→ R'G'B'（0〜1 に収める）．"""
    return np.clip((Y - off601 / 255) @ np.linalg.inv(T601 / 255).T, 0, 1)


names = ["赤", "緑", "青", "白", "黒"]
P = np.array([[255, 0, 0], [0, 255, 0], [0, 0, 255], [255, 255, 255], [0, 0, 0]]).T
Yp = np.floor(A @ P + b + 0.5).astype(int)   # 四捨五入（値は正）
print("P =\n", P)
print("Yp =\n", Yp)
check = np.floor(255 * rgb2ycbcr(P.T / 255) + 0.5).astype(int)   # BT.601 の定義で計算して一致するか
assert np.array_equal(check.T, Yp)
assert np.array_equal(Yp, [[81, 145, 41, 235, 16], [90, 54, 240, 128, 128], [240, 34, 110, 128, 128]])

スライドの表（代表的な色の変換）は，この結果から表全体を作る。無彩色（$x_\mathrm{R'}=x_\mathrm{G'}=x_\mathrm{B'}$）は色差が $128$（零に相当）になるので強調する（ここでは * を付ける）。白と黒は輝度の範囲の両端 235 と 16 になる。

In [ ]:
isAchrom = np.all(P == P[0], axis=0)          # 無彩色の列
print(f"{'':>22}" + "".join(f"{nm:>14}" for nm in names))
print(f"{'(xR, xG, xB)':>22}" + "".join(f"{'(%d,%d,%d)' % tuple(p):>15}" for p in P.T))
print(f"{'(yY, yCb, yCr)':>22}" + "".join(
    f"{('*(%d,%d,%d)' if a else '(%d,%d,%d)') % tuple(q):>15}" for q, a in zip(Yp.T, isAchrom)))

逆変換（Y'CbCr $\to$ R'G'B'）の行列は $\boldsymbol{A}^{-1}$ として計算できる。小数第 4 位に丸めると教科書の逆変換の行列と一致する（0 の成分は丸め誤差程度の小さな値になる）。

In [ ]:
Ainv = np.linalg.inv(A)
np.set_printoptions(precision=4, suppress=False)
print("Ainv =\n", Ainv)
np.set_printoptions(precision=4, suppress=True)
Ainv4 = np.round(Ainv, 4) + 0.0               # + 0.0 で -0 を 0 にする
print("Ainv4 =\n", Ainv4)
Ainv_book = np.array([[1.1644, 0, 1.5960], [1.1644, -0.3918, -0.8130], [1.1644, 2.0172, 0]])
assert np.array_equal(Ainv4, Ainv_book)       # 教科書の値と一致するか
Ainv3 = np.round(Ainv, 3) + 0.0
print("Ainv3 =\n", Ainv3)

逆変換で元に戻ることも確かめる。

In [ ]:
Pr = np.round(Ainv @ (Yp - b)).astype(int) + 0
print("Pr =\n", Pr)

Y'CbCr を 8-bit 整数に丸めているため，±1 程度の誤差が残ることがある（0〜255 の範囲外に出ることもあるので，実際にはクリッピングする）。

## YCbCr の成分画像
Y は明るさ（ほぼグレースケール画像），Cb と Cr は色の差だけを持つ。色差成分は変化が緩やかである。

In [ ]:
Ycc = rgb2ycbcr(Xrgb)
fig, axs = plt.subplots(1, 3, figsize=(12, 2.9), layout="constrained")
for k, ttl in enumerate(["Y", "Cb", "Cr"]):
    show(axs[k], Ycc[:, :, k], ttl)
plt.show()

## 色差サブサンプリング
人の眼は輝度より色の変化に鈍感なので，色差の標本化間隔を広げてデータ量を減らす。4:2:0 形式では Cb, Cr を縦横 1/2 に間引く。縮小・拡大の実装が MATLAB と異なるため，PSNR の値はスライドとわずかに異なる。

In [ ]:
Cb = resize(rescale(Ycc[:, :, 1], 0.5, order=1, anti_aliasing=True), (N1, N2), order=1)
Cr = resize(rescale(Ycc[:, :, 2], 0.5, order=1, anti_aliasing=True), (N1, N2), order=1)
X420 = ycbcr2rgb(np.stack([Ycc[:, :, 0], Cb, Cr], axis=2))
psnr420 = peak_signal_noise_ratio(Xrgb, X420, data_range=1)   # ピーク値 1 の PSNR
print(f"psnr420 = {psnr420:.4f} dB（{psnr420:.1f} dB）")

fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), layout="constrained")
show(axs[0], Xrgb, "4:4:4")
show(axs[1], X420, "4:2:0")
plt.show()

一画素当たりのビット数とビットレート（$1080\times1920$ 画素，$\Delta_\mathrm{t}=1/30$ s，各成分 8 bit）。4:2:0 形式では $R=\Delta_\mathrm{t}^{-1}N_1N_2(\beta_\mathrm{Y}+\beta_\mathrm{Cb}/4+\beta_\mathrm{Cr}/4)$。4:2:0 は前回の演習課題（SMPTE295M の 4:2:0 形式，約 746 Mbps）と同じ値になる。

In [ ]:
fmt = ["4:4:4", "4:2:2", "4:2:0"]
bpp = np.array([8 + 8 + 8, 8 + 8 / 2 + 8 / 2, 8 + 8 / 4 + 8 / 4])
Rfmt = bpp * Nv * Nh * fr
print(f"{'形式':<6} {'bpp':>4} {'R [Mbps]':>10}")
for f_, bp, R_ in zip(fmt, bpp, Rfmt):
    print(f"{f_:<8} {bp:>4.0f} {R_ / 1e6:>10.4f}")
assert np.array_equal(bpp, [24, 16, 12])

## CMY 空間
各成分を 0〜1 に正規化すると，CMY は RGB の補色：$\vec{y} = \vec{1}-\vec{x}$（教科書の例題「CMY空間」）。オレンジ色の画素 $\vec{x}=(1\ 0.5\ 0)^\top$ で確かめる。

In [ ]:
xo = np.array([1, 0.5, 0])
ycmy = 1 - xo
print("ycmy =", ycmy)
assert np.array_equal(ycmy, [0, 0.5, 1])

## CMYK 空間
まず RGB の補色 $(1-x_\mathrm{R},1-x_\mathrm{G},1-x_\mathrm{B})$ の最小値として $y_\mathrm{K}$ を求め，K で補正した CMY を $\vec{1} - \vec{x}/(1-y_\mathrm{K})$ で求める。途中の値（補色と $1-y_\mathrm{K}$）も表示する。

In [ ]:
xk = np.array([0.2, 0.4, 0.6])
xkc = 1 - xk                                  # RGB の補色
yK = xkc.min()
yCMY = 1 - xk / (1 - yK)
xback = (1 - yK) * (1 - yCMY)                 # 逆変換（教科書の例題の解答）で元に戻る
print("xkc =", xkc)
print("yK =", yK, " 1 - yK =", 1 - yK)
print("yCMY =", yCMY)
print("xback =", xback)
assert np.allclose(xkc, [0.8, 0.6, 0.4]) and np.isclose(yK, 0.4)
assert np.array_equal(np.round(yCMY, 3), [0.667, 0.333, 0.0]) and np.allclose(xback, xk)

## HS 系：色相環
色相（H）を角度，彩度（S）を中心からの距離で表す。赤 0°，緑 120°，青 240°。明度 V=1 の断面を描く。

In [ ]:
n = 301
c0 = (n - 1) / 2                              # 中心の添字
rr, cc = np.mgrid[0:n, 0:n]
dx = (cc - c0) / c0
dy = -(rr - c0) / c0                          # 上向きを正にする
H = np.mod(np.arctan2(dy, dx) / (2 * np.pi), 1)
S = np.minimum(np.hypot(dx, dy), 1)
W = color.hsv2rgb(np.stack([H, S, np.ones((n, n))], axis=2))
W[np.hypot(dx, dy) > 1] = 1                   # 円の外は白
fig, ax = plt.subplots(figsize=(3.5, 3.5), layout="constrained")
show(ax, W)
plt.show()

## RGB 各成分と HSV 各成分
教科書の HSV 変換（scikit-image の `color.rgb2hsv`）で成分画像を作る（講義のデモ hsidemo と同じ見せ方）。

In [ ]:
Xhsv = color.rgb2hsv(Xrgb)
fig, axs = plt.subplots(1, 3, figsize=(12, 2.9), layout="constrained")
for k, ttl in enumerate(["H", "S", "V"]):
    show(axs[k], Xhsv[:, :, k], ttl)
plt.show()

数値例：オレンジ $\vec{x}=(1\ 0.5\ 0)^\top$。教科書の式どおりに計算する。$x_\mathrm{max}=1,\ x_\mathrm{min}=0$ より $y_\mathrm{V}=x_\mathrm{max}=1,\ y_\mathrm{S}=(x_\mathrm{max}-x_\mathrm{min})/x_\mathrm{max}=1$。R が最大で G が最小でないので $y_\mathrm{H} = (60^\circ - 60^\circ\cdot(x_\mathrm{max}-x_\mathrm{G})/(x_\mathrm{max}-x_\mathrm{min}))/360^\circ = 30^\circ/360^\circ$。

In [ ]:
xmx, xmn = xo.max(), xo.min()
yV = xmx
yS = (xmx - xmn) / xmx
Hdeg = 60 - 60 * (xmx - xo[1]) / (xmx - xmn)  # [度]
print(f"xmax = {xmx:g}，xmin = {xmn:g}，yV = {yV:g}，yS = {yS:g}，H = {Hdeg:.0f} 度")
hsvo = color.rgb2hsv(xo)                      # ライブラリの関数と一致するか
print("hsvo =", hsvo)
assert np.max(np.abs(hsvo - [Hdeg / 360, yS, yV])) < 1e-12
assert (Hdeg, yS, yV) == (30, 1, 1)

## インデックス方式
前回スライドの例：$4\times4$ 画素，4 色（$\beta_\mathrm{I}=2$ bit），カラーマップの各色は RGB 各 2 bit（計 6 bit）。

In [ ]:
idx = np.array([[0, 0, 0, 0], [0, 1, 1, 2], [3, 1, 1, 1], [3, 3, 3, 3]])
map6 = ["001111", "111100", "110101", "110011"]          # RGB 各 2 bit
cmap = np.array([[int(m[0:2], 2), int(m[2:4], 2), int(m[4:6], 2)] for m in map6]) / 3
print("idx =\n", idx)
print("cmap =\n", cmap)
Xidx = cmap[idx]                              # インデックス 0 がマップの 0 行目
fig, ax = plt.subplots(figsize=(3, 3), layout="constrained")
show(ax, Xidx)
plt.show()

総ビット数を比べる。インデックス方式は $B=\beta_\mathrm{I}N_1N_2 + 2^{\beta_\mathrm{I}}(\beta_\mathrm{R}+\beta_\mathrm{G}+\beta_\mathrm{B})$（第 2 項はカラーマップ分）。

In [ ]:
betaI, bitsMap = 2, 6
N1e, N2e = idx.shape
Bidx1 = betaI * N1e * N2e                     # インデックス配列の分
Bidx2 = 2 ** betaI * bitsMap                  # カラーマップの分
Bidx = Bidx1 + Bidx2
Brgb44 = bitsMap * N1e * N2e                  # RGB カラー方式
print(f"Bidx1 = {Bidx1}，Bidx2 = {Bidx2}，Bidx = {Bidx}，Brgb44 = {Brgb44}")
assert (Bidx1, Bidx2, Bidx, Brgb44) == (32, 24, 56, 96)

## インデックス方式の実画像例
写真を 8 色（$\beta_\mathrm{I}=3$）に減色する。色数は限られるが，一画素 3 bit で済む。

Pillow の `quantize`（メディアンカット法，ディザなし）で 8 色を選ぶ。減色の方法が MATLAB 版と異なるため，選ばれる 8 色はスライドとわずかに異なる。

In [ ]:
img = Image.fromarray(img_as_ubyte(Xrgb))
q = img.quantize(colors=8, method=Image.Quantize.MEDIANCUT, dither=Image.Dither.NONE)
I8 = np.asarray(q)                            # インデックス 0〜7
map8 = np.array(q.getpalette()[:3 * 8]).reshape(8, 3) / 255
X8i = map8[I8]
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), layout="constrained",
                        gridspec_kw=dict(width_ratios=[3, 1]))
show(axs[0], X8i, "8 色（インデックス）")
show(axs[1], map8[None, :, :], "カラーマップ")
plt.show()

In [ ]:
Bind8 = 3 * N1 * N2 + 2 ** 3 * 24
ratio = Brgb / Bind8
print(f"Bind8 = {Bind8:,} bits，ratio = {ratio:.4f}（{ratio:.1f} 倍）")

## まとめ
- 量子化 $y=\lfloor x/Q \rfloor$ で輝度値を離散化する。階調が少ないと擬似輪郭が現れる
- データ量は $B=\beta N_1N_2$ [bits]，映像なら $R=\beta N_1N_2\Delta_\mathrm{t}^{-1}$ [bps]
- XYZ への変換行列は三原色と白色点の色度から決まる。YCbCr は輝度と色差に分け，色差サブサンプリングでデータ量を減らせる
- CMY は RGB の補色，CMYK は黒を加えた 4 成分，HSV は色相・彩度・明度
- インデックス方式は色を限定する代わりに一画素のビット数を減らす

© Copyright, Shogo MURAMATSU, All rights reserved.